## ***IMPORT LIBRERIE***


In [1]:
import os
import pandas as pd
import numpy as np
from app_config import REPORTS_DIR, FIGURES_DIR, MODELS_DIR
import sliding_window_on_data
from torch.utils.data import DataLoader
import glob

from models.DeepConvLSTM import DeepConvLSTM, HARDataset 
import optuna
from optuna.pruners import MedianPruner
import optuna.visualization as vis
import train
import torch
import torch.nn as nn
import train_with_cm
import matplotlib.pyplot as plt
#definisco il path da cui leggere i .csv

from utils.log_config import logger
from figures import plot_CM

#definisco il path da cui leggere i .csv

path='C:\codes\HumanActivityRecognition\data\pdd_data'
logger.debug(path)

2025-05-04 14:54:40.941 | INFO     | app_config:<module>:11 - PROJ_ROOT path is: C:\codes\HumanActivityRecognition
2025-05-04 14:54:53,544 - INFO - myapp - Logging configured from C:\codes\HumanActivityRecognition\HumanActivityRecognition\utils\base_config.json
2025-05-04 14:55:02,993 - INFO - myapp - GPU not available, training on CPU.
c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-04 14:55:03,985 - DEBUG - matplotlib - matplotlib data path: c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\matplotlib\mpl-data
2025-05-04 14:55:04,015 - DEBUG - matplotlib - CONFIGDIR=C:\Users\carol\.matplotlib
2025-05-04 14:55:04,098 - DEBUG - matplotlib - interactive is False
2025-05-04 14:55:04,104 - DEBUG - matplotlib - platform is win32
2025-05-04 14:55:04,273 

## ***DATA PREPROCESSING***

*scansiono recording e filtro per righe non nulle*  
*OUTPUT: unico dataframe con tutte le attività non nulle*

In [4]:
#Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
#al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
#appendo tutte le righe delle righe non nulle in un unico dataframe
#per tutti i file che terminano in .csv nella cartella path
# Definisco il percorso della cartella contenente i CSV

# Nome del file CSV finale
final_csv_path = os.path.join(path, 'df_DO_non_null.csv')

# Controllo se il file esiste già
if os.path.exists(final_csv_path):
    logger.debug(f"Il file {final_csv_path} esiste già. Lo sto caricando...")
    df_doll = pd.read_csv(final_csv_path)
else:
    #trovo tutti i file che corrispondono a "DO" nella cartella path e li stampo a schermo
    files = glob.glob(os.path.join(path, "*_DO*.csv"))
    logger.debug(f"Files: {files}")
    
    kid_doll, kid_doll_no_null = [], [] # liste per salvare utenti prima e dopo il merge 
    df_list_doll = [] # lista vuota per appendere i dataframe con attività non nulla

    for file in files:
        df = pd.read_csv(file)
        logger.debug(f"Original shape: {df.shape}")
        kid_doll.append(df['kid_id'].unique())
        df = df[df['action_id'] != 0] #filtro le righe con action_id non nullo
        logger.debug(f"Filtered shape: {df.shape}")
        kid_doll_no_null.append(df['kid_id'].unique())

        logger.debug(f"Columns: {df.columns}")
        logger.debug(f"Action counts:\n{df['action'].value_counts()}")
        logger.debug(f"Toy counts:\n{df['toy_id'].value_counts()}")
        logger.debug("="*50)

    # mi stampo gli utenti prima di fare il merge e dopo il merge
    logger.info(f"Numero di utenti che hanno fatto almeno un azione: {len(kid_doll_no_null)/len(kid_doll)}")

    '''
    Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
    al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
    appendo tutte le righe non nulle in un unico dataframe per tutti i file che terminano in .csv nella balltella path
    '''

    for file in os.listdir(path):
        if not file.endswith('.csv'):
            continue
    
        #leggo solo i file che dopo l'undescore ha BA*.csv
        if file.split('_')[-1].startswith('DO') and file.endswith('.csv'): #controllo che il file termini con .csv
            df_temp=pd.read_csv(os.path.join(path,file))
            df_temp = df_temp[df_temp['action_id'] != 0]
            df_list_doll.append(df_temp)

    df_doll = pd.concat(df_list_doll)
    logger.info(f"Shape finale del dataframe: {df_doll.shape}")
    logger.info(f"Colonne del dataframe: {list(df_doll.columns)}")
    logger.info(f"Conteggio delle azioni:\n{df_doll['action'].value_counts()}")

    # salvo il dataframe
    df_doll.to_csv(os.path.join(path,'df_DO_non_null.csv'),index=False)

2025-05-04 15:01:51,862 - DEBUG - myapp - Files: ['C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3002_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3003_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3005_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3006_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3007_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3008_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3009_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3010_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3011_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3013_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3017_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3018_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3019_DO.csv', 'C:\\codes\\HumanActivityRecognition\\data\\pdd_data\\3020_DO.csv'

C:\Users\carol\AppData\Local\Temp\ipykernel_8120\4283014396.py:23: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
2025-05-04 15:01:52,894 - DEBUG - myapp - Original shape: (169262, 25)
2025-05-04 15:01:52,915 - DEBUG - myapp - Filtered shape: (389, 25)
2025-05-04 15:01:52,922 - DEBUG - myapp - Columns: Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
2025-05-04 15:01:52,927 - DEBUG - myapp - Action counts:
action
sposta posto    389
Name: count, dtype: int64
2025-05-04 15:01:52,931 - DEBUG - myapp - Toy counts:
toy_id
DO    389
Name: count, dtype: int64
2025-05-04 15:01:52,934 - DEBUG -

*per ogni attività trovata, salvo un .csv*  
*OUTPUT: un .csv per ogni attività non nulla (df_doll_action_11.csv, df_doll_action_19.csv ecc... )*


In [5]:
#ora divido il dataframe in base all'attività (action_id) e salvo i dataframe in un file .csv
#per ogni attività

for action_id in df_doll['action_id'].unique():
    df_action = df_doll[df_doll["action_id"] == action_id] #filtro il dataframe in base all'attività
    logger.debug(f"Dimensioni del dataframe df_doll_action_{action_id} - {df_action.shape}") #log delle dimensioni del dataframe
    logger.info(f"Conteggio delle attività per df_doll_action_{action_id}") #log del conteggio delle attività
    #salvo il dataframe
    df_action.to_csv(os.path.join(path,f'df_doll_action_{action_id}.csv'),index=False) #index=False per non salvare l'indice
    logger.debug(f"Salvato il dataframe df_doll_action_{action_id}.csv")

2025-05-04 15:04:22,534 - DEBUG - myapp - Dimensioni del dataframe df_doll_action_41 - (1562, 25)
2025-05-04 15:04:22,536 - INFO - myapp - Conteggio delle attività per df_doll_action_41


2025-05-04 15:04:22,585 - DEBUG - myapp - Salvato il dataframe df_doll_action_41.csv
2025-05-04 15:04:22,589 - DEBUG - myapp - Dimensioni del dataframe df_doll_action_3 - (2307, 25)
2025-05-04 15:04:22,591 - INFO - myapp - Conteggio delle attività per df_doll_action_3
2025-05-04 15:04:22,659 - DEBUG - myapp - Salvato il dataframe df_doll_action_3.csv
2025-05-04 15:04:22,665 - DEBUG - myapp - Dimensioni del dataframe df_doll_action_31 - (239, 25)
2025-05-04 15:04:22,667 - INFO - myapp - Conteggio delle attività per df_doll_action_31
2025-05-04 15:04:22,686 - DEBUG - myapp - Salvato il dataframe df_doll_action_31.csv
2025-05-04 15:04:22,690 - DEBUG - myapp - Dimensioni del dataframe df_doll_action_4 - (650, 25)
2025-05-04 15:04:22,693 - INFO - myapp - Conteggio delle attività per df_doll_action_4
2025-05-04 15:04:22,718 - DEBUG - myapp - Salvato il dataframe df_doll_action_4.csv
2025-05-04 15:04:22,721 - DEBUG - myapp - Dimensioni del dataframe df_doll_action_34 - (1882, 25)
2025-05-04 1

## ***DATA PROCESSING***

*SLIDING WINDOW*  
*OUTPUT: unica matrice con tutte le windows concatenate*

In [7]:
#applico sliding window con la funzion process_csv
nb_sensor_channels = 9
sliding_window_length = 100
sliding_window_step = 50

#ora applico la funzione sliding window (che mi da come output x_window e y_window) a tutti i .csv relativi al giocattolo doll
#e poi concateno tutto in un unica x e y 

X, Y = [], []
kid_action_counts = {}

for action_file in [f for f in os.listdir(path) if f.endswith('.csv') and f.split('_')[1] == 'doll']:
    file_path = os.path.join(path, action_file)
    action = action_file.split('_')[-1].split('.')[0]

    X_windows, Y_windows, kid_id_action_dict = sliding_window_on_data.process_csv(file_path, nb_sensor_channels, sliding_window_length, sliding_window_step)

    X.append(X_windows)
    Y.append(Y_windows)


    logger.info(f"Numero  totale di finestre per l'azione {action}:{len(X_windows)}")
    kid_action_counts[f"doll_action_{action}"] = kid_id_action_dict #aggiungo il dizionario al dizionario principale per tenere traccia del numero di finestre per ogni bambino per ogni azione, ogni doll_action è una chiave e il valore è un dizionario con il numero di finestre per ogni bambino
    logger.info(f"Contenuto finale di kid_action_counts: {kid_action_counts}") #per veere quante finestre per ogni azione e per ogni bambino sono state elaborte 


# Concateno tutti i dati in un unico array per X e Y
X = np.concatenate(X, axis=0)
Y = np.concatenate(Y, axis=0)


# Stampo le dimensioni di X e Y
logger.info(f"Dimensioni di X finale: {X.shape}")
logger.info(f"Dimensioni di Y finale: {Y.shape}")

2025-05-04 19:36:00,586 - DEBUG - myapp - sto leggendo il file csv: C:\codes\HumanActivityRecognition\data\pdd_data\df_doll_action_10.csv
2025-05-04 19:36:00,600 - INFO - myapp - Kid_id: 3003, X_kid shape: (798, 9), Y_kid shape: (798,)
2025-05-04 19:36:00,605 - INFO - myapp - Numero di finestre estratte: 14
2025-05-04 19:36:00,608 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 48
2025-05-04 19:36:00,612 - INFO - myapp - Padding normale applicato. Padding code: 1
2025-05-04 19:36:00,616 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 9)
2025-05-04 19:36:00,618 - INFO - myapp - Numero totale di finestre (dopo padding finale): 15
2025-05-04 19:36:00,621 - DEBUG - myapp - X_windows shape after sliding window: (15, 100, 9)
2025-05-04 19:36:00,623 - DEBUG - myapp - Padding codes: [1]
2025-05-04 19:36:00,623 - INFO - myapp - Numero di finestre estratte: 14
2025-05-04 19:36:00,624 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 48
2025-05-04 

>>> Kid_ids: [3003 3005 3006 3008 3009 3011 3013 3017 3022]


2025-05-04 19:36:00,771 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 1)
2025-05-04 19:36:00,773 - INFO - myapp - Numero totale di finestre (dopo padding finale): 8
2025-05-04 19:36:00,774 - DEBUG - myapp - Y_windows_full shape after sliding window: (8, 100, 1)
2025-05-04 19:36:00,776 - DEBUG - myapp - Padding codes: [1]
2025-05-04 19:36:00,779 - DEBUG - myapp - Y_windows shape after extraction: (8, 1)
2025-05-04 19:36:00,781 - INFO - myapp - Kid_id: 3008, Action_id: 10.0, Action_count: 8
2025-05-04 19:36:00,788 - INFO - myapp - Kid_id: 3009, X_kid shape: (335, 9), Y_kid shape: (335,)
2025-05-04 19:36:00,798 - INFO - myapp - Numero di finestre estratte: 5
2025-05-04 19:36:00,800 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 35
2025-05-04 19:36:00,803 - INFO - myapp - Padding normale applicato. Padding code: 1
2025-05-04 19:36:00,805 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 9)
2025-05-04 19:36:00,808 - INFO - myapp - Numero totale di finest

>>> Kid_ids: [3009 3010 3013 3023]


2025-05-04 19:36:01,221 - DEBUG - myapp - Padding codes: [1]
2025-05-04 19:36:01,223 - DEBUG - myapp - Y_windows shape after extraction: (1, 1)
2025-05-04 19:36:01,229 - INFO - myapp - Kid_id: 3013, Action_id: 11.0, Action_count: 1
2025-05-04 19:36:01,238 - INFO - myapp - Kid_id: 3023, X_kid shape: (175, 9), Y_kid shape: (175,)
2025-05-04 19:36:01,242 - INFO - myapp - Numero di finestre estratte: 2
2025-05-04 19:36:01,245 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 25
2025-05-04 19:36:01,249 - INFO - myapp - Padding estremo applicato. Padding code: 2
2025-05-04 19:36:01,251 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 9)
2025-05-04 19:36:01,254 - INFO - myapp - Numero totale di finestre (dopo padding finale): 3
2025-05-04 19:36:01,258 - DEBUG - myapp - X_windows shape after sliding window: (3, 100, 9)
2025-05-04 19:36:01,260 - DEBUG - myapp - Padding codes: [2]
2025-05-04 19:36:01,263 - INFO - myapp - Numero di finestre estratte: 2
2025-05-04 19:36:0

...

>>> Kid_ids: [3003 3011 3017]
>>> Kid_ids: [3017]
>>> Kid_ids: [3003]
>>> Kid_ids: [3022]


2025-05-04 19:36:02,130 - INFO - myapp - Numero totale di finestre (dopo padding finale): 13
2025-05-04 19:36:02,132 - DEBUG - myapp - X_windows shape after sliding window: (13, 100, 9)
2025-05-04 19:36:02,132 - DEBUG - myapp - Padding codes: [2]
2025-05-04 19:36:02,133 - INFO - myapp - Numero di finestre estratte: 12
2025-05-04 19:36:02,134 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 26
2025-05-04 19:36:02,135 - INFO - myapp - Padding estremo applicato. Padding code: 2
2025-05-04 19:36:02,136 - DEBUG - myapp - Nuova finestra con padding: (1, 100, 1)
2025-05-04 19:36:02,136 - INFO - myapp - Numero totale di finestre (dopo padding finale): 13
2025-05-04 19:36:02,137 - DEBUG - myapp - Y_windows_full shape after sliding window: (13, 100, 1)
2025-05-04 19:36:02,138 - DEBUG - myapp - Padding codes: [2]
2025-05-04 19:36:02,139 - DEBUG - myapp - Y_windows shape after extraction: (13, 1)
2025-05-04 19:36:02,140 - INFO - myapp - Kid_id: 3022, Action_id: 32.0, Action_co

>>> Kid_ids: [3003]
>>> Kid_ids: [3003 3011 3017 3022]
>>> Kid_ids: [3002 3006 3011]


2025-05-04 19:36:02,348 - INFO - myapp - Numero totale di finestre (dopo padding finale): 7
2025-05-04 19:36:02,349 - DEBUG - myapp - Y_windows_full shape after sliding window: (7, 100, 1)
2025-05-04 19:36:02,349 - DEBUG - myapp - Padding codes: [1]
2025-05-04 19:36:02,350 - DEBUG - myapp - Y_windows shape after extraction: (7, 1)
2025-05-04 19:36:02,352 - INFO - myapp - Kid_id: 3002, Action_id: 41.0, Action_count: 7
2025-05-04 19:36:02,356 - INFO - myapp - Kid_id: 3006, X_kid shape: (60, 9), Y_kid shape: (60,)
2025-05-04 19:36:02,359 - DEBUG - myapp - La lunghezza della finestra è maggiore della lunghezza dell'array. Applico padding.
2025-05-04 19:36:02,361 - INFO - myapp - Padding iniziale: 14, Padding finale: 26
2025-05-04 19:36:02,364 - INFO - myapp - Nuova lunghezza array: 100
2025-05-04 19:36:02,366 - INFO - myapp - Numero di finestre estratte: 1
2025-05-04 19:36:02,368 - DEBUG - myapp - Campioni rimanenti dopo l'ultima finestra completa: 0
2025-05-04 19:36:02,369 - INFO - myapp 

>>> Kid_ids: [3009 3022]
>>> Kid_ids: [3009 3022]
>>> Kid_ids: [3009 3022]


In [8]:
from collections import defaultdict

kid_summary = defaultdict(dict)

for action, kid_counts in kid_action_counts.items():
    for kid, count in kid_counts.items():
        kid_summary[kid][action] = count

# Stampo il riepilogo per ogni bambino
for kid, actions in kid_summary.items():
    logger.info(f"Bambino {kid}:")
    for action, count in actions.items():
        logger.info(f"  {action}: {count} finestre")
    logger.info("="*50)

2025-05-05 11:25:00,794 - INFO - myapp - Bambino 3003:
2025-05-05 11:25:00,799 - INFO - myapp -   doll_action_10: 15 finestre
2025-05-05 11:25:00,801 - INFO - myapp -   doll_action_16: 1 finestre
2025-05-05 11:25:00,804 - INFO - myapp -   doll_action_27: 1 finestre
2025-05-05 11:25:00,807 - INFO - myapp -   doll_action_3: 20 finestre
2025-05-05 11:25:00,809 - INFO - myapp -   doll_action_31: 4 finestre
2025-05-05 11:25:00,811 - INFO - myapp -   doll_action_34: 37 finestre
2025-05-05 11:25:00,814 - INFO - myapp -   doll_action_4: 1 finestre
2025-05-05 11:25:00,815 - INFO - myapp - ==================================================
2025-05-05 11:25:00,818 - INFO - myapp - Bambino 3005:
2025-05-05 11:25:00,819 - INFO - myapp -   doll_action_10: 18 finestre
2025-05-05 11:25:00,820 - INFO - myapp -   doll_action_12: 2 finestre
2025-05-05 11:25:00,820 - INFO - myapp -   doll_action_16: 1 finestre
2025-05-05 11:25:00,823 - INFO - myapp - ==================================================
2025

*SPLIT DATASET IN TRS, VS,TS*  
*OUTPUT: array NumPy di TRS/VS/TS (X e Y)*

*RIASSEGNAZIONE DELLE ETICHETTE*

## ***OPTUNA***

*nella funzione obiettivo: uso train e val*  
*stampo cm di train e val*  
*train finale con dati di test con cm*


